# Raw training-data inspection

This notebook validates and combines downloaded PSV records into `data/physionet/combined.parquet` when needed. Descriptive summaries and patient-level trend comparisons use source-A training records only; the source-shift table separately compares source-A training measurements with unlabeled source-B measurements. Source-A labels are used to reproduce the patient-level split (seed 42) and define label-negative event windows. Raw values are not imputed, scaled, clipped, or removed for summaries. A small diagnostic classifier comparison uses fold-local imputation only inside cross-validation; it is not clinical model validation.

Use the project's `.venv` kernel after installing the package with `python -m pip install -e .`. Downloaded records, Parquet data, and patient-level plots belong under the Git-ignored `data/` directory; do not commit them. Structural or schema errors stop the scan. Extreme values are review candidates, not automatically errors.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from matplotlib import pyplot as plt

notebook_path = globals().get('__vsc_ipynb_file__')
current = Path(notebook_path).resolve().parent if notebook_path else Path.cwd().resolve()
project_root = next(
    (
        path for path in (current, *current.parents)
        if (path / 'pyproject.toml').is_file()
        and (path / 'src' / 'sepsis_warning').is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError('Open this notebook from the SUTD-PML-team2-main project')

# Make the checkout's src-layout package available in the notebook kernel.
source_dir = str(project_root / 'src')
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
from sepsis_warning.data import DYNAMIC, STATIC, Patient, split_patients
from sepsis_warning.prepare import prepare_parquet

data_root = project_root / 'data'
data_dir = data_root / 'physionet'
data_file = data_dir / 'combined.parquet'
if not data_file.is_file():
    raw_records = sum(
        len(list((data_dir / f'training_set{hospital}').glob('*.psv')))
        for hospital in ('A', 'B')
    )
    if not raw_records:
        raise FileNotFoundError(
            f'No downloaded PSV records found under {data_dir}. '
            'Download records first with: '
            'python scripts/download_data.py --limit 500 --output data/physionet'
        )
    print('Combined Parquet is missing; validating and combining downloaded records...')
    preparation_summary = prepare_parquet(data_dir, data_file)
    print(
        f"Combined {preparation_summary['records']:,} records and "
        f"{preparation_summary['hourly_rows']:,} hourly rows."
    )

dataset = ds.dataset(data_file, format='parquet')
required_columns = {'patient_id', 'hospital', 'SepsisLabel', *DYNAMIC, *STATIC}
missing_columns = required_columns - set(dataset.schema.names)
if missing_columns:
    raise ValueError(f'Combined Parquet is missing required columns: {sorted(missing_columns)}')

# Read source-A labels only to reproduce the same stratified patient splits.
source_a_labels = dataset.to_table(
    columns=['patient_id', 'hospital', 'SepsisLabel'],
    filter=ds.field('hospital') == 'A',
)
source_a_summary = source_a_labels.group_by(
    ['hospital', 'patient_id']
).aggregate([
    ('SepsisLabel', 'max'),
    ('SepsisLabel', 'min'),
])
source_patients = [
    Patient(
        patient_id=f"{row['hospital']}:{row['patient_id']}",
        hospital=row['hospital'],
        frame=pd.DataFrame({'SepsisLabel': [int(row['SepsisLabel_max'])]}),
    )
    for row in sorted(source_a_summary.to_pylist(), key=lambda item: item['patient_id'])
]
# split_patients requires both sources; this placeholder is never used for selection.
source_patients.append(
    Patient('B:external-placeholder', 'B', pd.DataFrame({'SepsisLabel': [0]}))
)
splits = split_patients(source_patients, seed=42)
training_records = splits['train']
training_keys = {
    (record.hospital, record.patient_id.split(':', 1)[1])
    for record in training_records
}
if not training_keys:
    raise ValueError('No source-A training records available')
training_ids = sorted(patient_id for _, patient_id in training_keys)
# Prefer a septic stay that was not already positive at its first recorded hour.
eligible_examples = [
    (row['hospital'], row['patient_id'])
    for row in source_a_summary.to_pylist()
    if (row['hospital'], row['patient_id']) in training_keys
    and row['SepsisLabel_max'] == 1
    and row['SepsisLabel_min'] == 0
]
example_key = min(eligible_examples) if eligible_examples else min(training_keys)
plot_columns = ('HR', 'MAP', 'Temp', 'Resp')
feature_columns = [*DYNAMIC, *STATIC]
observed_values = {column: [] for column in plot_columns}
stats = {
    column: {'missing': 0, 'observed': 0, 'minimum': None, 'maximum': None}
    for column in feature_columns
}
record_lengths = {key: 0 for key in training_keys}
trajectory = []
hour_count = positive_hours = 0

# The scanner predicate excludes validation, calibration, internal-test and B rows.
scanner = dataset.scanner(
    columns=['patient_id', 'hospital', *feature_columns, 'SepsisLabel'],
    filter=(ds.field('hospital') == 'A') & ds.field('patient_id').isin(training_ids),
    batch_size=65_536,
)
for batch in scanner.to_batches():
    hours = batch.to_pandas()
    hour_count += len(hours)
    positive_hours += int(hours['SepsisLabel'].sum())
    for patient_id, count in hours.groupby('patient_id', sort=False).size().items():
        key = ('A', patient_id)
        if key not in record_lengths:
            raise ValueError(f'Unexpected non-training record in scan: {key}')
        record_lengths[key] += int(count)
    for column in feature_columns:
        values = hours[column].to_numpy(dtype=np.float64)
        finite = np.isfinite(values)
        if np.any(~finite & ~np.isnan(values)):
            raise ValueError(f'Nonfinite observed value in {column}')
        feature = stats[column]
        feature['missing'] += int(np.isnan(values).sum())
        feature['observed'] += int(finite.sum())
        if finite.any():
            minimum, maximum = float(values[finite].min()), float(values[finite].max())
            feature['minimum'] = minimum if feature['minimum'] is None else min(feature['minimum'], minimum)
            feature['maximum'] = maximum if feature['maximum'] is None else max(feature['maximum'], maximum)
            if column in observed_values:
                observed_values[column].extend(values[finite].tolist())
    example_rows = hours.loc[
        (hours['hospital'] == example_key[0])
        & (hours['patient_id'] == example_key[1]),
        [*plot_columns, 'ICULOS', 'SepsisLabel'],
    ]
    trajectory.extend(example_rows.to_dict(orient='records'))

if hour_count != sum(record_lengths.values()) or any(length == 0 for length in record_lengths.values()):
    raise ValueError('Training Parquet rows do not match the selected training records')
lengths = list(record_lengths.values())
missingness = {
    column: 100 * value['missing'] / hour_count
    for column, value in stats.items()
}
print(f'Source-A training only: {len(training_records):,} records; {hour_count:,} hours')
print(f'Positive-label hours: {positive_hours:,} ({100 * positive_hours / hour_count:.2f}%)')
print(f'Example training record: {example_key[0]}:{example_key[1]}')
print('Raw values retained; no imputation or model training.')
print(f"{'Feature':<20} {'Missing %':>10} {'Observed':>10} {'Minimum':>14} {'Maximum':>14}")
for column in sorted(feature_columns, key=missingness.get, reverse=True):
    value = stats[column]
    print(f"{column:<20} {missingness[column]:>10.2f} {value['observed']:>10,} "
          f"{str(value['minimum']):>14} {str(value['maximum']):>14}")

## Understand the training data and cohort charts

The counts, missingness rates, and record lengths below are calculated from the local combined Parquet each time this notebook runs. Record counts are ICU stays, not necessarily unique people. Source A is split at the record level into approximately 55% training, 15% calibration, 15% validation, and 15% internal test using the project's `split_patients` function and seed 42. Source B is external test; it is not used in patient-level cohort trends and is used only for the unlabeled measurement-shift comparison in the final section.

### 1. Hourly labels and record lengths

- The first chart shows counts of supplied zero and one labels for source-A training hours.
- The record-length histogram counts each training record once. Recorded hours are not necessarily complete ICU stays.
- The positive label is already shifted six hours early and persists afterward. Do not shift it again. Accuracy alone is misleading for this imbalanced outcome.

### 2. Feature missingness

- Bars show the percentage of missing training-hour values for each input used by this project.
- A longer bar means fewer measurements were available, not that values were more abnormal. Missingness alone does not establish corruption or justify dropping a feature.
- No missing values are filled in for this inspection; fold-local median imputation is limited to the diagnostic classifier comparison described in the final section.

### 3. Raw-value distributions

The four histograms show observed training values for HR (beats/minute), MAP (mmHg), Temp (degrees Celsius), and Resp (breaths/minute). The logarithmic count axis helps show tails; extreme values are candidates for review, not automatically errors. No values are clipped or removed.

### 4. Illustrative record trajectory

The final figure plots a deterministic source-A training stay. Dots show observed measurements; missing values remain gaps. The dashed line marks the first positive supplied label, not clinical onset or a model alert. This single record is not a representative cohort summary. Keep its patient-level chart local and clear notebook outputs before sharing or committing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
axes[0].bar(['Label 0', 'Label 1'], [hour_count - positive_hours, positive_hours])
axes[0].set(title='Training: supplied hourly labels', ylabel='Hourly observations')
axes[1].hist(lengths, bins=40, edgecolor='white')
axes[1].set(title='Training: full record-length range', xlabel='Recorded hours', ylabel='Records')
plt.show()

items = sorted(missingness.items(), key=lambda item: item[1])
fig, ax = plt.subplots(figsize=(10, 11), layout='constrained')
ax.barh([column for column, _ in items], [percent for _, percent in items])
ax.set(xlim=(0, 100), xlabel='Missing hourly values (%)', title='Training: original missingness')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout='constrained')
for ax, column in zip(axes.flat, plot_columns, strict=True):
    values = observed_values[column]
    if values:
        ax.hist(values, bins=60, edgecolor='white')
        ax.set_yscale('log')
    else:
        ax.text(0.5, 0.5, 'No observed values', ha='center', transform=ax.transAxes)
    ax.set(title=f'{column}: observed values, full range', xlabel=column, ylabel='Count (log scale)')
fig.suptitle('Training: inspect tails and extremes; no clipping or filling')
plt.show()


## 5. One illustrative ICU-stay trajectory

The next figure has five vertically aligned panels: **HR**, **MAP**, **Temp**, **Resp**, and the **supplied sepsis label**. The measurements use the units listed above; the label is either 0 or 1.

- **Shared X-axis:** original `ICULOS`, the recorded ICU-hour index. The first available hour need not be 1.
- Dots mark observed hourly values. Lines connect adjacent observed hours; missing measurements remain gaps, not zeros or filled estimates.
- **Interpretation:** inspect sudden jumps, extreme values, sustained changes and long measurement gaps. None is automatically classified as an error.
- The bottom step plot shows the supplied label over time. The red dashed line marks its **first positive hour**, not clinical onset or a model alert. The label is a target, never an input feature.
- The notebook chooses the first sorted training record that is ever-positive but not positive in its first recorded row. If none exists, it uses the first training record. This selection avoids an ambiguous first-row-positive example when possible, but is not a representative sample of the cohort.
- Changes near the label boundary do not establish causation or prove early-warning performance. This illustrative chart is descriptive; the separate diagnostic cross-validation is not clinical model validation.

**Keep patient-level trajectories local and clear outputs before committing or sharing.**

In [ ]:
icu_hours = [row['ICULOS'] for row in trajectory]
positive_times = [row['ICULOS'] for row in trajectory if row['SepsisLabel'] == 1]
fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True, layout='constrained')
units = ('beats/min', 'mmHg', 'degrees C', 'breaths/min')
for ax, column, unit in zip(axes[:4], plot_columns, units, strict=True):
    values = [float('nan') if row[column] is None else row[column] for row in trajectory]
    ax.plot(icu_hours, values, marker='.', linewidth=1)
    ax.set_ylabel(f'{column}\n({unit})')
    ax.grid(alpha=0.2)
axes[4].step(icu_hours, [row['SepsisLabel'] for row in trajectory], where='post')
axes[4].set(yticks=[0, 1], ylabel='Supplied label', xlabel='Original ICULOS')
if positive_times:
    for ax in axes:
        ax.axvline(positive_times[0], linestyle='--', color='red')
fig.suptitle('One illustrative training record: raw values, missingness remains gaps')
plt.show()
print('Dashed line: first positive supplied label, NOT clinical onset or a model alert.')
print('This deterministic example is not representative of the entire cohort.')


## Prospective comparisons before the event boundary

These comparisons use source-A training stays only. An event stay is indexed at its first positive supplied label; because the challenge label is shifted six hours before its dataset-defined onset, this is a label boundary, not a clinically verified onset. Each event window contains only the 24 label-negative hours before that boundary (approximately 7–30 hours before dataset-defined onset if the six-hour shift is exact). The first positive hour and every later hour are excluded. Event stays without 24 earlier hours are left-censored/short-history exclusions.

Each non-event stay receives a deterministic pseudo-index matched to the event index-hour distribution. It is included only when a complete 24-hour history is observed before that pseudo-index. This aligns time-in-stay and follow-up opportunity without treating later observations as prospective data. The resulting cohorts are not paired or clinically matched.

### Measurement trends and cadence

The heatmaps show the fraction of patient-hours with an observed value for every dynamic measurement, separately by outcome and relative hour. The trend panels show raw observed medians for six common measurements; gaps reflect absent measurements, and no uncertainty intervals are shown. Cadence differences can change which values contribute to these medians, so interpret both figures together.

### Source shift

The table compares source-A training measurements with all source-B measurements without reading source-B labels. It reports missingness percentage-point differences and observed-value mean differences/standardized mean differences. These are descriptive dataset shifts, not external validation; source B has previously been inspected and is not an untouched confirmatory cohort.

### Missingness diagnostic

Three logistic-regression baselines are compared by stratified patient-level cross-validation within source-A training: 24-hour missingness fractions only, observed-value medians only, and both together. All inputs are derived exclusively from each patient's pre-index window. Median imputation is fitted inside each training fold only. ROC AUC and average precision are exploratory, especially with few eligible events; stronger missingness-only performance would show that observation patterns carry outcome information in this sample, not that missingness causally drives a clinical prediction. This is not a validated model or a prospective performance estimate.

Patient-level plots and results should remain local. Clear notebook outputs before sharing or committing.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

analysis_filter = (ds.field('hospital') == 'A') & ds.field('patient_id').isin(training_ids)
label_scanner = dataset.scanner(
    columns=['patient_id', 'ICULOS', 'SepsisLabel'],
    filter=analysis_filter,
    batch_size=65_536,
)
first_positive_hour = {}
record_start_hour = {}
record_end_hour = {}
for batch in label_scanner.to_batches():
    labels = batch.to_pandas()
    for patient_id, stay in labels.groupby('patient_id', sort=False):
        start_hour = float(stay['ICULOS'].min())
        end_hour = float(stay['ICULOS'].max())
        record_start_hour[patient_id] = min(record_start_hour.get(patient_id, start_hour), start_hour)
        record_end_hour[patient_id] = max(record_end_hour.get(patient_id, end_hour), end_hour)
        positive_hours_for_patient = stay.loc[stay['SepsisLabel'] == 1, 'ICULOS']
        if not positive_hours_for_patient.empty:
            first_hour = float(positive_hours_for_patient.min())
            first_positive_hour[patient_id] = min(
                first_positive_hour.get(patient_id, first_hour), first_hour
            )

# Exclude events without a full 24-hour label-negative history (left-censored or short stays).
event_index = {
    patient_id: index_hour
    for patient_id, index_hour in first_positive_hour.items()
    if index_hour - record_start_hour[patient_id] >= 24
}
if not event_index:
    raise ValueError('No source-A training events have 24 pre-label hours available')
control_ids = sorted(set(training_ids) - set(first_positive_hour))
event_index_hours = np.asarray(sorted(event_index.values()), dtype=float)
control_index_hours = np.quantile(
    event_index_hours,
    (np.arange(len(control_ids)) + 0.5) / max(len(control_ids), 1),
    method='nearest',
).astype(int) if control_ids else np.array([], dtype=int)
analysis_index = dict(event_index)
for patient_id, index_hour in zip(control_ids, control_index_hours, strict=True):
    has_history = index_hour - record_start_hour[patient_id] >= 24
    observed_through_index = record_end_hour[patient_id] >= index_hour - 1
    if has_history and observed_through_index:
        analysis_index[patient_id] = int(index_hour)

analysis_ids = sorted(analysis_index)
window_scanner = dataset.scanner(
    columns=['patient_id', 'ICULOS', 'SepsisLabel', *DYNAMIC],
    filter=(ds.field('hospital') == 'A') & ds.field('patient_id').isin(analysis_ids),
    batch_size=65_536,
)
window_parts = []
for batch in window_scanner.to_batches():
    hours = batch.to_pandas()
    hours['index_hour'] = hours['patient_id'].map(analysis_index)
    hours['lead_hours'] = hours['index_hour'] - hours['ICULOS']
    window = hours.loc[hours['lead_hours'].between(1, 24)].copy()
    if not window.empty:
        if window['SepsisLabel'].any():
            raise ValueError('A positive-label row entered the prospective pre-label window')
        window['relative_hour'] = -window['lead_hours'].astype(int)
        window['outcome'] = np.where(window['patient_id'].isin(event_index), 'Event', 'Non-event')
        window_parts.append(window)
window_rows = pd.concat(window_parts, ignore_index=True)
window_counts = window_rows.groupby('patient_id').size()
if set(window_counts.index) != set(analysis_index) or not window_counts.eq(24).all():
    raise ValueError('Each analyzed stay must contribute exactly 24 pre-index label-negative hours')

sample_values = window_rows.groupby('patient_id')[DYNAMIC].median().add_prefix('value__')
missing_inputs = window_rows[['patient_id', *DYNAMIC]].copy()
missing_inputs[DYNAMIC] = missing_inputs[DYNAMIC].isna().astype(float)
sample_missingness = missing_inputs.groupby('patient_id')[DYNAMIC].mean().add_prefix('missing__')
samples = sample_values.join(sample_missingness).reset_index()
samples['event'] = samples['patient_id'].isin(event_index).astype(int)
left_censored_events = len(first_positive_hour) - len(event_index)
matched_controls = int((samples['event'] == 0).sum())
print(
    f"Prospective windows: {len(event_index)} events and {matched_controls} matched non-events; "
    f"excluded {left_censored_events} left-censored/short-history events."
)
print('Each window contains only the 24 label-negative hours before its index; no index or later rows are used.')
print('Controls are matched on index-hour distribution and must be observed through the hour before the pseudo-index.')

# Measurement cadence by feature and hour relative to the index; -1 is the last included hour.
long_measurements = window_rows.melt(
    id_vars=['patient_id', 'outcome', 'relative_hour'],
    value_vars=DYNAMIC,
    var_name='feature',
    value_name='value',
)
long_measurements['observed'] = long_measurements['value'].notna()
cadence = (
    long_measurements.groupby(['outcome', 'relative_hour', 'feature'])['observed']
    .mean()
    .rename('observed_rate')
    .reset_index()
)
cadence_summary = (
    long_measurements.groupby(['outcome', 'feature'])['observed']
    .mean()
    .unstack('outcome')
    .reindex(DYNAMIC)
)
availability = 100 * cadence_summary[['Event', 'Non-event']]
availability['Event minus non-event (pp)'] = availability['Event'] - availability['Non-event']
print('\nMeasurement availability over the 24-hour pre-index window (% patient-hours observed):')
print(availability.round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(18, 9), layout='constrained', sharey=True)
relative_hours = list(range(-24, 0))
for ax, outcome in zip(axes, ('Event', 'Non-event'), strict=True):
    matrix = (
        cadence.loc[cadence['outcome'] == outcome]
        .pivot(index='feature', columns='relative_hour', values='observed_rate')
        .reindex(index=DYNAMIC, columns=relative_hours)
    )
    image = ax.imshow(matrix.to_numpy(), aspect='auto', vmin=0, vmax=1, cmap='viridis')
    ax.set(title=f'{outcome}: measurement availability', xlabel='Hours before index')
    ax.set_xticks(np.arange(len(relative_hours)), labels=relative_hours)
    ax.set_yticks(np.arange(len(DYNAMIC)), labels=DYNAMIC)
axes[0].set_ylabel('Measurement')
fig.colorbar(image, ax=axes, label='Fraction of patient-hours observed')
plt.show()

trend_features = [name for name in ('HR', 'MAP', 'Temp', 'Resp', 'O2Sat', 'Lactate') if name in DYNAMIC]
fig, axes = plt.subplots(2, 3, figsize=(14, 8), layout='constrained', sharex=True)
for ax, feature in zip(axes.flat, trend_features, strict=True):
    for outcome in ('Event', 'Non-event'):
        subset = window_rows.loc[window_rows['outcome'] == outcome]
        trend = subset.groupby('relative_hour')[feature].median()
        ax.plot(trend.index, trend.values, marker='.', label=outcome)
    ax.set(title=feature, xlabel='Hours before first positive label / matched pseudo-index', ylabel='Observed median')
    ax.set_xlim(-24, 0)
    ax.grid(alpha=0.2)
axes[0, 0].legend()
fig.suptitle('Raw measurements before the index only; missing values are not imputed')
plt.show()

# Source comparison is measurement-only: source-A training stays versus all source-B stays.
source_filter = (ds.field('hospital') == 'B') | (
    (ds.field('hospital') == 'A') & ds.field('patient_id').isin(training_ids)
)
source_scanner = dataset.scanner(
    columns=['hospital', 'patient_id', *DYNAMIC],
    filter=source_filter,
    batch_size=65_536,
)
source_stats = {
    hospital: {
        feature: {'rows': 0, 'observed': 0, 'sum': 0.0, 'sum_sq': 0.0}
        for feature in DYNAMIC
    }
    for hospital in ('A', 'B')
}
source_rows = {'A': 0, 'B': 0}
source_patients = {'A': set(), 'B': set()}
for batch in source_scanner.to_batches():
    hours = batch.to_pandas()
    for hospital, group in hours.groupby('hospital', sort=False):
        source_rows[hospital] += len(group)
        source_patients[hospital].update(group['patient_id'].unique())
        for feature in DYNAMIC:
            values = group[feature].to_numpy(dtype=np.float64)
            observed = values[np.isfinite(values)]
            stats_for_feature = source_stats[hospital][feature]
            stats_for_feature['rows'] += len(values)
            stats_for_feature['observed'] += len(observed)
            stats_for_feature['sum'] += float(observed.sum())
            stats_for_feature['sum_sq'] += float(np.square(observed).sum())

source_shift_rows = []
for feature in DYNAMIC:
    values_by_source = {}
    for hospital in ('A', 'B'):
        feature_stats = source_stats[hospital][feature]
        count = feature_stats['observed']
        mean = feature_stats['sum'] / count if count else np.nan
        variance = max(feature_stats['sum_sq'] / count - mean ** 2, 0) if count else np.nan
        missing_percent = 100 * (1 - count / feature_stats['rows']) if feature_stats['rows'] else np.nan
        values_by_source[hospital] = (mean, variance, missing_percent, count)
    mean_a, variance_a, missing_a, observed_a = values_by_source['A']
    mean_b, variance_b, missing_b, observed_b = values_by_source['B']
    pooled_sd = np.sqrt((variance_a + variance_b) / 2)
    standardized_difference = (mean_b - mean_a) / pooled_sd if pooled_sd > 0 else np.nan
    source_shift_rows.append({
        'Feature': feature,
        'A missing %': missing_a,
        'B missing %': missing_b,
        'Missingness shift (pp)': missing_b - missing_a,
        'Mean difference (B - A)': mean_b - mean_a,
        'Standardized mean difference': standardized_difference,
        'A observed': observed_a,
        'B observed': observed_b,
    })
source_shift = pd.DataFrame(source_shift_rows).set_index('Feature')
print(
    f"\nMeasurement-only source comparison: A training {len(source_patients['A']):,} stays / "
    f"{source_rows['A']:,} hours; B {len(source_patients['B']):,} stays / {source_rows['B']:,} hours."
)
print('B labels are not read. These descriptive shifts do not constitute external validation.')
print(source_shift.round(2).to_string())

# Patient-level cross-validation of label-negative 24-hour summaries only.
value_columns = [f'value__{feature}' for feature in DYNAMIC]
missing_columns = [f'missing__{feature}' for feature in DYNAMIC]
feature_sets = {
    'Missingness only': missing_columns,
    'Observed values only': value_columns,
    'Values + missingness': value_columns + missing_columns,
}
class_counts = samples['event'].value_counts()
folds = min(5, int(class_counts.min()))
if folds < 2:
    print('\nMissingness diagnostic skipped: fewer than two eligible patients in one outcome class.')
else:
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    scores = []
    for name, columns in feature_sets.items():
        diagnostic = make_pipeline(
            SimpleImputer(strategy='median', keep_empty_features=True),
            StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight='balanced'),
        )
        probabilities = cross_val_predict(
            diagnostic, samples[columns], samples['event'], cv=cv, method='predict_proba'
        )[:, 1]
        scores.append({
            'Diagnostic features': name,
            'ROC AUC': roc_auc_score(samples['event'], probabilities),
            'Average precision': average_precision_score(samples['event'], probabilities),
        })
    print(f'\nExploratory {folds}-fold patient-level CV on {len(samples)} source-A training stays:')
    print(pd.DataFrame(scores).set_index('Diagnostic features').round(3).to_string())
    print('Fold-local median imputation is used only for classifier inputs; this is not clinical model validation.')

### Individual pre-index time series

Each faint line represents one eligible stay; the bold lines show the event and non-event medians at each relative hour. Event stays are aligned to their first positive supplied label, while non-events use matched pseudo-indexes. The plot includes only the 24 label-negative hours before each index (hours -24 through -1); missing measurements remain gaps, and patient IDs are not shown. Event counts are small, so individual traces and medians are descriptive rather than inferential. Keep these patient-level traces local.

In [ ]:
stay_colors = {'Event': '#176b87', 'Non-event': '#d97824'}
fig, axes = plt.subplots(2, 3, figsize=(14, 8), layout='constrained', sharex=True)
for ax, feature in zip(axes.flat, trend_features, strict=True):
    for outcome, color in stay_colors.items():
        subset = window_rows.loc[window_rows['outcome'] == outcome]
        for _, stay in subset.groupby('patient_id', sort=True):
            stay = stay.sort_values('relative_hour')
            ax.plot(
                stay['relative_hour'], stay[feature],
                color=color, alpha=0.18, linewidth=0.8,
            )
        median = subset.groupby('relative_hour')[feature].median()
        ax.plot(
            median.index, median.values,
            color=color, marker='.', linewidth=2.4, label=f'{outcome} median',
        )
    ax.set(title=feature, xlabel='Hours before first positive label / matched pseudo-index', ylabel='Observed value')
    ax.set_xlim(-24, 0)
    ax.grid(alpha=0.2)
axes[0, 0].plot([], [], color=stay_colors['Event'], alpha=0.35, label='Event stays')
axes[0, 0].plot([], [], color=stay_colors['Non-event'], alpha=0.35, label='Non-event stays')
axes[0, 0].legend(fontsize='small')
fig.suptitle('Pre-index time series: individual stays (faint) and cohort medians (bold)')
plt.show()

### Multivariate structure with PCA

The vital-sign time-series panels show six selected measurements, while the availability heatmaps cover all dynamic measurements. These PCA plots summarize the higher-dimensional patient-level data: one point per stay, using the 24 label-negative pre-index hours. The panels separately project observed-value medians, missingness fractions, and both together; the additional 3D plot shows the combined profile. In this run, 30 value dimensions and 31 missingness dimensions varied across stays; constant dimensions are omitted.

Value medians are median-imputed for this visualization and rank-normalized feature by feature to reduce the influence of extreme values. Missingness fractions are standardized. PCA is fit without outcome labels; event status only determines point color/shape. Axis percentages show the variance represented by each component. These projections are descriptive, not a significance test or prediction result; overlap is expected with this small event cohort.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import QuantileTransformer, StandardScaler

value_columns = [f'value__{feature}' for feature in DYNAMIC]
missing_columns = [f'missing__{feature}' for feature in DYNAMIC]
value_profile = samples[value_columns]
active_value_columns = [
    column for column in value_columns
    if value_profile[column].nunique(dropna=True) > 1
]
active_missing_columns = [
    column for column in missing_columns
    if samples[column].nunique(dropna=True) > 1
]
if len(active_value_columns) < 3 or len(active_missing_columns) < 3:
    raise ValueError('Need at least three varying value and missingness features for PCA')

value_matrix = SimpleImputer(strategy='median').fit_transform(
    value_profile[active_value_columns]
)
value_matrix = QuantileTransformer(
    n_quantiles=min(50, len(samples)),
    output_distribution='normal',
    random_state=42,
).fit_transform(value_matrix)
missingness_matrix = StandardScaler().fit_transform(
    samples[active_missing_columns]
)
combined_matrix = np.column_stack((value_matrix, missingness_matrix))
pca_inputs = {
    'Observed values': value_matrix,
    'Missingness': missingness_matrix,
    'Values + missingness': combined_matrix,
}

print(
    f"PCA inputs: {len(active_value_columns)} varying value features, "
    f"{len(active_missing_columns)} varying missingness features, "
    f"{len(samples)} stays. Values are rank-normalized for outlier resistance."
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), layout='constrained')
class_styles = {
    0: {'label': 'Non-event', 'color': '#d97824', 'marker': 'o'},
    1: {'label': 'Event', 'color': '#176b87', 'marker': '^'},
}
for ax, (name, matrix) in zip(axes, pca_inputs.items(), strict=True):
    reducer = PCA(n_components=2, svd_solver='full')
    coordinates = reducer.fit_transform(matrix)
    for event, style in class_styles.items():
        selected = samples['event'].to_numpy() == event
        ax.scatter(
            coordinates[selected, 0], coordinates[selected, 1],
            label=style['label'], color=style['color'], marker=style['marker'],
            edgecolors='white', linewidths=0.5, s=52, alpha=0.9,
        )
    variance = reducer.explained_variance_ratio_
    ax.set(
        title=name,
        xlabel=f'PC1 ({variance[0]:.1%} variance)',
        ylabel=f'PC2 ({variance[1]:.1%} variance)',
    )
    ax.grid(alpha=0.2)
axes[0].legend()
fig.suptitle('PCA of pre-index, patient-level summaries (one point per stay)')
plt.show()

combined_pca_3d = PCA(n_components=3, svd_solver='full')
coordinates_3d = combined_pca_3d.fit_transform(combined_matrix)
fig = plt.figure(figsize=(8, 6), layout='constrained')
ax = fig.add_subplot(111, projection='3d')
for event, style in class_styles.items():
    selected = samples['event'].to_numpy() == event
    ax.scatter(
        coordinates_3d[selected, 0], coordinates_3d[selected, 1], coordinates_3d[selected, 2],
        label=style['label'], color=style['color'], marker=style['marker'],
        edgecolors='white', linewidths=0.4, s=48, alpha=0.9,
    )
variance_3d = combined_pca_3d.explained_variance_ratio_
ax.set_xlabel(f'PC1 ({variance_3d[0]:.1%})')
ax.set_ylabel(f'PC2 ({variance_3d[1]:.1%})')
ax.set_zlabel(f'PC3 ({variance_3d[2]:.1%})')
ax.set_title('Combined value and missingness profiles: 3D PCA')
ax.legend()
plt.show()

In [ ]:
requested_patient_id = 'p000014'
if ('A', requested_patient_id) in record_lengths:
    selected_patient_id = requested_patient_id
else:
    selected_patient_id = example_key[1]
    print(
        f'{requested_patient_id} is outside the source-A training split; '
        f'using the training example {selected_patient_id} instead.'
    )

# Count rows per source-record key without reading labels from either source.
length_scanner = dataset.scanner(
    columns=['patient_id', 'hospital', 'ICULOS'],
    batch_size=65_536,
)
all_record_summary = {}
scanned_hours = 0
for batch in length_scanner.to_batches():
    hour_rows = batch.to_pandas()
    scanned_hours += len(hour_rows)
    for (hospital, patient_id), stay in hour_rows.groupby(
        ['hospital', 'patient_id'], sort=False
    ):
        key = (hospital, patient_id)
        summary = all_record_summary.setdefault(
            key,
            {'recorded_hours': 0, 'first_iculos': np.inf, 'last_iculos': -np.inf},
        )
        summary['recorded_hours'] += len(stay)
        summary['first_iculos'] = min(summary['first_iculos'], float(stay['ICULOS'].min()))
        summary['last_iculos'] = max(summary['last_iculos'], float(stay['ICULOS'].max()))
if scanned_hours != sum(value['recorded_hours'] for value in all_record_summary.values()):
    raise ValueError('Scanned hourly rows do not match per-record duration totals')

all_stay_lengths = pd.DataFrame([
    {
        'hospital': hospital,
        'patient_id': patient_id,
        **summary,
    }
    for (hospital, patient_id), summary in all_record_summary.items()
])
record_counts_per_key = all_stay_lengths.groupby(['hospital', 'patient_id']).size()
print(f'All downloaded data: {len(all_stay_lengths):,} source-record keys; {scanned_hours:,} hourly rows.')
print(
    'Records per (source, patient_id) key: '
    f'{record_counts_per_key.value_counts().sort_index().to_dict()}'
)
print('In this file-per-record dataset, each source-record key corresponds to one record.')

source_colors = {'A': '#176b87', 'B': '#d97824'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout='constrained')
for hospital, color in source_colors.items():
    source_stays = (
        all_stay_lengths.loc[all_stay_lengths['hospital'] == hospital]
        .sort_values(['recorded_hours', 'patient_id'], kind='stable')
        .reset_index(drop=True)
    )
    source_stays['rank'] = np.arange(1, len(source_stays) + 1)
    axes[0].scatter(
        source_stays['rank'], source_stays['recorded_hours'],
        label=f'Source {hospital} ({len(source_stays)} records)',
        color=color, alpha=0.7, s=22,
    )
    axes[1].hist(
        source_stays['recorded_hours'], bins=32, alpha=0.55,
        color=color, label=f'Source {hospital}',
    )
    selected_stay = source_stays.loc[
        source_stays['patient_id'] == selected_patient_id
    ] if hospital == 'A' else source_stays.iloc[0:0]
    if not selected_stay.empty:
        axes[0].scatter(
            selected_stay['rank'], selected_stay['recorded_hours'],
            label=f'Displayed stay ({selected_patient_id})',
            marker='*', color='#222222', edgecolors='white', linewidths=0.5,
            s=180, zorder=3,
        )
axes[0].set(
    xlabel='Record rank within source, sorted by observed hours',
    ylabel='Recorded hourly rows',
    title='Observed duration for each downloaded record',
)
axes[1].set(
    xlabel='Recorded hourly rows', ylabel='Number of records',
    title='Observed duration distribution by source',
)
for axis in axes:
    axis.grid(alpha=0.2)
axes[0].legend()
axes[1].legend()
fig.suptitle('Recorded coverage, not necessarily complete ICU length of stay')
plt.show()

patient_rows = dataset.to_table(
    columns=['patient_id', 'hospital', 'ICULOS', *DYNAMIC, 'SepsisLabel'],
    filter=(ds.field('hospital') == 'A') & (ds.field('patient_id') == selected_patient_id),
).to_pandas().sort_values('ICULOS')
if patient_rows.empty:
    raise ValueError(f'No source-A training rows found for {selected_patient_id}')
print(
    f"Displayed stay A:{selected_patient_id}: {len(patient_rows)} recorded hours, "
    f"ICULOS {patient_rows['ICULOS'].min():g}-{patient_rows['ICULOS'].max():g}. "
    'Recorded coverage may be shorter than the full ICU stay.'
)

features_with_label = [*DYNAMIC, 'SepsisLabel']
fig, axes = plt.subplots(7, 5, figsize=(18, 20), sharex=True, layout='constrained')
for axis, feature in zip(axes.flat, features_with_label, strict=True):
    if feature == 'SepsisLabel':
        axis.step(
            patient_rows['ICULOS'], patient_rows[feature],
            where='post', color='#a83232', linewidth=1.2,
        )
        axis.set_yticks([0, 1])
    else:
        axis.plot(
            patient_rows['ICULOS'],
            patient_rows[feature].to_numpy(dtype=np.float64),
            marker='.', linewidth=0.8, markersize=2.5,
        )
    axis.set_title(feature)
    axis.grid(alpha=0.2)
for axis in axes[-1, :]:
    axis.set_xlabel('ICULOS (hour since ICU admission)')
fig.supylabel('Raw value (feature-specific scale)')
fig.suptitle(
    f'A:{selected_patient_id}: raw hourly measurements and supplied label; gaps remain missing'
)
plt.show()

### Per-stay duration and selected-record measurements

The duration plot has one point per source-A training ICU record, sorted by recorded hourly-row count and colored by whether its supplied label is ever positive. The star marks the record shown below. These are observed record lengths, not guaranteed complete ICU lengths of stay, and a record is not necessarily a unique person. The measurement panels show every dynamic variable for the selected training record; axes use feature-specific raw scales, and missing values remain gaps. Edit `requested_patient_id` in the next cell to inspect another source-A training record. If the requested ID is outside the training split, the notebook uses its existing deterministic training example instead.

### Age and gender by supplied-label status

Each point represents one source-A training ICU record. Age and gender are reduced to one value per record so longer stays do not contribute extra points. Panels separate the dataset's gender codes (0 = female, 1 = male); points and boxplots compare age distributions for records with and without any positive supplied label. Age 100 is the dataset's code for age 90 or older. The supplied label is shifted six hours early and is not a clinically verified onset. This is a descriptive cohort comparison, not a causal or inferential result.

In [ ]:
demographic_scanner = dataset.scanner(
    columns=['patient_id', 'Age', 'Gender'],
    filter=(ds.field('hospital') == 'A') & ds.field('patient_id').isin(training_ids),
    batch_size=65_536,
)
demographic_parts = [batch.to_pandas() for batch in demographic_scanner.to_batches()]
if not demographic_parts:
    raise ValueError('No source-A training demographics found')
demographic_hours = pd.concat(demographic_parts, ignore_index=True)
if demographic_hours.groupby('patient_id')[['Age', 'Gender']].nunique(dropna=False).gt(1).any().any():
    raise ValueError('Age or Gender changes within a source-A training record')
demographics = (
    demographic_hours.groupby('patient_id', sort=True)
    .agg(Age=('Age', 'first'), Gender=('Gender', 'first'))
    .reset_index()
)
if demographics[['Age', 'Gender']].isna().any().any():
    raise ValueError('Missing Age or Gender in source-A training records')
if not set(demographics['Gender'].unique()).issubset({0, 1}):
    raise ValueError('Unexpected Gender code; expected 0 (female) or 1 (male)')

label_by_patient = {
    row['patient_id']: int(row['SepsisLabel_max'])
    for row in source_a_summary.to_pylist()
    if ('A', row['patient_id']) in training_keys
}
demographics['ever_positive_label'] = demographics['patient_id'].map(label_by_patient)
if demographics['ever_positive_label'].isna().any():
    raise ValueError('Could not match every training record to its supplied-label summary')
demographics['ever_positive_label'] = demographics['ever_positive_label'].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True, layout='constrained')
outcome_styles = {
    0: {'label': 'No positive label', 'color': '#d97824'},
    1: {'label': 'Ever-positive label', 'color': '#176b87'},
}
random_generator = np.random.default_rng(42)
for axis, gender_code, gender_label in zip(
    axes, (0, 1), ('Female', 'Male'), strict=True
):
    gender_records = demographics.loc[demographics['Gender'] == gender_code]
    ticks = []
    tick_labels = []
    for outcome, style in outcome_styles.items():
        group = gender_records.loc[
            gender_records['ever_positive_label'] == outcome, 'Age'
        ].to_numpy(dtype=np.float64)
        position = outcome + 1
        count = len(group)
        ticks.append(position)
        tick_labels.append(f"{style['label']}\n(n={count})")
        if count:
            axis.boxplot(
                group, positions=[position], widths=0.48,
                showfliers=False, patch_artist=True,
                boxprops={'facecolor': style['color'], 'alpha': 0.18},
                medianprops={'color': style['color'], 'linewidth': 2},
                whiskerprops={'color': style['color']},
                capprops={'color': style['color']},
            )
            jitter = random_generator.uniform(-0.09, 0.09, size=count)
            axis.scatter(
                position + jitter, group, color=style['color'],
                alpha=0.72, s=28, edgecolors='white', linewidths=0.35,
            )
    axis.set(
        title=f'{gender_label} records (n={len(gender_records)})',
        xticks=ticks, xticklabels=tick_labels,
        xlim=(0.45, 2.55), xlabel='Supplied-label status for the full record',
    )
    axis.grid(axis='y', alpha=0.2)
axes[0].set_ylabel('Age (100 denotes age 90 or older)')
fig.suptitle('Age by gender and ever-positive supplied-label status\nSource-A training records only')
plt.show()
print(
    demographics.groupby(['Gender', 'ever_positive_label']).size()
    .rename('records').to_string()
)

### All input features by gender and pre-index label group

This grid expands the age comparison to all 34 physiological inputs plus Age, HospAdmTime, and median ICULOS within each eligible pre-index window. Gender is used to form panels within each feature, and colors distinguish non-event records from eligible event records. Each point is one stay; for dynamic measurements it is the observed median over the 24 label-negative hours before the index. Missing values are not imputed, and the per-group observed count is shown under each category. `Gender` is the grouping variable; `SepsisLabel`, IDs, source, and excluded administrative unit codes are not plotted as predictors. Event counts are small, so this is exploratory and descriptive.

In [ ]:
eligible_ids = sorted(samples['patient_id'].unique())
static_scanner = dataset.scanner(
    columns=['patient_id', 'Age', 'Gender', 'HospAdmTime'],
    filter=(ds.field('hospital') == 'A') & ds.field('patient_id').isin(eligible_ids),
    batch_size=65_536,
)
static_parts = [batch.to_pandas() for batch in static_scanner.to_batches()]
if not static_parts:
    raise ValueError('No static features found for eligible pre-index stays')
static_rows = pd.concat(static_parts, ignore_index=True)
static_columns = ['Age', 'Gender', 'HospAdmTime']
if static_rows.groupby('patient_id')[static_columns].nunique(dropna=False).gt(1).any().any():
    raise ValueError('A static feature changes within an eligible stay')
static_by_patient = (
    static_rows.groupby('patient_id', sort=True)[static_columns]
    .first()
    .reset_index()
)
window_iculos = (
    window_rows.groupby('patient_id')['ICULOS']
    .median()
    .rename('ICULOS')
    .reset_index()
)
feature_profiles = (
    samples[['patient_id', 'event', *value_columns]]
    .merge(static_by_patient, on='patient_id', validate='one_to_one')
    .merge(window_iculos, on='patient_id', validate='one_to_one')
)
feature_names = [*DYNAMIC, 'Age', 'HospAdmTime', 'ICULOS']
profile_column = {feature: f'value__{feature}' for feature in DYNAMIC}
profile_column.update({
    'Age': 'Age',
    'HospAdmTime': 'HospAdmTime',
    'ICULOS': 'ICULOS',
})

plot_groups = [
    {'gender': 0, 'event': 0, 'short': 'F no', 'color': '#d97824', 'marker': 'o'},
    {'gender': 0, 'event': 1, 'short': 'F event', 'color': '#176b87', 'marker': 'o'},
    {'gender': 1, 'event': 0, 'short': 'M no', 'color': '#d97824', 'marker': '^'},
    {'gender': 1, 'event': 1, 'short': 'M event', 'color': '#176b87', 'marker': '^'},
]
columns_per_row = 6
rows = int(np.ceil(len(feature_names) / columns_per_row))
fig, axes = plt.subplots(
    rows, columns_per_row, figsize=(22, rows * 2.7),
    layout='constrained', squeeze=False,
)
random_generator = np.random.default_rng(42)
for axis, feature in zip(axes.flat, feature_names, strict=False):
    column = profile_column[feature]
    tick_labels = []
    for position, group in enumerate(plot_groups, start=1):
        selected = feature_profiles.loc[
            (feature_profiles['Gender'] == group['gender'])
            & (feature_profiles['event'] == group['event']),
            column,
        ].to_numpy(dtype=np.float64)
        observed = selected[np.isfinite(selected)]
        tick_labels.append(f"{group['short']}\n{len(observed)}")
        if len(observed) >= 3:
            axis.boxplot(
                observed, positions=[position], widths=0.55,
                showfliers=False, patch_artist=True,
                boxprops={'facecolor': group['color'], 'alpha': 0.16},
                medianprops={'color': group['color'], 'linewidth': 1.5},
                whiskerprops={'color': group['color']},
                capprops={'color': group['color']},
            )
        if len(observed):
            jitter = random_generator.uniform(-0.13, 0.13, size=len(observed))
            axis.scatter(
                position + jitter, observed,
                color=group['color'], marker=group['marker'],
                alpha=0.68, s=14, edgecolors='white', linewidths=0.25,
            )
    axis.set_title(feature, fontsize=9)
    axis.set_xticks(np.arange(1, len(plot_groups) + 1), labels=tick_labels)
    axis.tick_params(axis='x', labelsize=6, pad=1)
    axis.grid(axis='y', alpha=0.18)
for axis in axes.flat[len(feature_names):]:
    axis.set_visible(False)
fig.supylabel('Observed patient-level summary (feature-specific scale)')
event_records = int(feature_profiles['event'].sum())
fig.suptitle(
    f'{len(feature_names)} numeric model inputs stratified by gender '
    f'(n={len(feature_profiles)} eligible stays; {event_records} event stays)'
)
plt.show()
print(
    'Groups: F/M = female/male; no = no positive supplied label; event = ever-positive supplied label. '
    'Counts under each group are records with an observed value for that feature.'
)
print('Gender is shown as the grouping variable; IDs, source, target values, and Unit1/Unit2 are excluded.')

### Changes approaching the first positive label

For each eligible stay and feature, this comparison subtracts the median in hours -24 to -13 from the median in hours -12 to -1. Both halves are label-negative; no first-positive or post-positive measurements are included. The table reports changes in the feature's original units and the event-minus-control contrast scaled by that feature's pre-index IQR.

In this sample, the largest contrasts are Glucose (+1.11 IQR; 3 paired event stays), Phosphate (+0.93 IQR; 2), and WBC (+0.60 IQR; 1). HR also rises more in event stays (+7 beats/min median change across 9 paired stays), while SBP and MAP tend to fall. These are exploratory signals, not reliable rankings: each feature has its own smaller paired sample because labs are sparse, and there are only 9 eligible event stays overall. No significance tests are performed. The supplied label is already shifted six hours before dataset-defined onset, so these changes precede the first positive label, not a clinically verified onset.

In [ ]:
early_period = window_rows.loc[window_rows['relative_hour'].between(-24, -13)]
late_period = window_rows.loc[window_rows['relative_hour'].between(-12, -1)]
change_rows = []
for patient_id, stay in window_rows.groupby('patient_id', sort=True):
    outcome = stay['outcome'].iloc[0]
    early_stay = early_period.loc[early_period['patient_id'] == patient_id]
    late_stay = late_period.loc[late_period['patient_id'] == patient_id]
    for feature in DYNAMIC:
        early_values = early_stay[feature].dropna()
        late_values = late_stay[feature].dropna()
        if early_values.empty or late_values.empty:
            continue
        early_median = float(early_values.median())
        late_median = float(late_values.median())
        change_rows.append({
            'patient_id': patient_id,
            'outcome': outcome,
            'feature': feature,
            'early_median': early_median,
            'late_median': late_median,
            'change': late_median - early_median,
            'early_observations': len(early_values),
            'late_observations': len(late_values),
        })
if not change_rows:
    raise ValueError('No features have observations in both halves of any eligible stay')
paired_changes = pd.DataFrame(change_rows)

change_summary_rows = []
for feature in DYNAMIC:
    feature_changes = paired_changes.loc[paired_changes['feature'] == feature]
    event_changes = feature_changes.loc[feature_changes['outcome'] == 'Event']
    control_changes = feature_changes.loc[feature_changes['outcome'] == 'Non-event']
    observed_values_for_feature = window_rows[feature].dropna()
    feature_iqr = (
        float(observed_values_for_feature.quantile(0.75) - observed_values_for_feature.quantile(0.25))
        if not observed_values_for_feature.empty else np.nan
    )
    event_delta = float(event_changes['change'].median()) if not event_changes.empty else np.nan
    control_delta = float(control_changes['change'].median()) if not control_changes.empty else np.nan
    delta_difference = event_delta - control_delta
    change_summary_rows.append({
        'Feature': feature,
        'Event paired stays': event_changes['patient_id'].nunique(),
        'Event early median': event_changes['early_median'].median(),
        'Event late median': event_changes['late_median'].median(),
        'Event median change': event_delta,
        'Control paired stays': control_changes['patient_id'].nunique(),
        'Control median change': control_delta,
        'Event-control change / feature IQR': (
            delta_difference / feature_iqr if feature_iqr > 0 else np.nan
        ),
    })
change_summary = pd.DataFrame(change_summary_rows)
change_summary['absolute contrast'] = change_summary['Event-control change / feature IQR'].abs()
change_summary = change_summary.sort_values(
    'absolute contrast', ascending=False, na_position='last'
).drop(columns='absolute contrast')
print(
    'Within-stay change = median in hours -12 to -1 minus median in hours -24 to -13. '
    'Both periods are before the first positive label.'
)
print('Only stays with at least one observed value in each half contribute to a feature change.')
print(change_summary.round(2).to_string(index=False))

heatmap_columns = [
    'Event median change',
    'Control median change',
    'Event-control change / feature IQR',
]
change_matrix = change_summary.set_index('Feature')[heatmap_columns].copy()
for feature in DYNAMIC:
    observed_values_for_feature = window_rows[feature].dropna()
    feature_iqr = (
        float(observed_values_for_feature.quantile(0.75) - observed_values_for_feature.quantile(0.25))
        if not observed_values_for_feature.empty else np.nan
    )
    if feature_iqr > 0:
        change_matrix.loc[feature, 'Event median change'] /= feature_iqr
        change_matrix.loc[feature, 'Control median change'] /= feature_iqr
change_matrix = change_matrix.reindex(change_summary['Feature'])
heatmap_values = np.ma.masked_invalid(change_matrix.to_numpy(dtype=np.float64))
color_map = plt.cm.RdBu_r.copy()
color_map.set_bad('#d9d9d9')
fig, ax = plt.subplots(figsize=(8, 11), layout='constrained')
image = ax.imshow(heatmap_values, aspect='auto', cmap=color_map, vmin=-1.5, vmax=1.5)
ax.set_yticks(np.arange(len(change_matrix.index)), labels=change_matrix.index)
ax.set_xticks(
    np.arange(len(heatmap_columns)),
    labels=['Event change / IQR', 'Control change / IQR', 'Event-control / IQR'],
)
for row_index in range(len(change_matrix.index)):
    for column_index in range(len(heatmap_columns)):
        value = change_matrix.iloc[row_index, column_index]
        label = f'{value:.2f}' if pd.notna(value) else 'n/a'
        ax.text(column_index, row_index, label, ha='center', va='center', fontsize=7)
ax.set_title('Pre-label within-stay feature changes (scaled by feature IQR)')
fig.colorbar(image, ax=ax, label='Change relative to feature IQR')
plt.show()

### Visualizing time since last measurement

This schematic follows one variable through a short example history. A recorded value makes its measurement age zero; each later unobserved hour increments that age. Before the variable has ever been observed, age is undefined and the `never_observed` indicator is 1. The example uses no patient data.

In [ ]:
example_hours = np.arange(1, 11)
example_observed = np.array([False, False, True, False, False, True, False, False, True, False])
example_values = np.array([np.nan, np.nan, 82.0, np.nan, np.nan, 91.0, np.nan, np.nan, 88.0, np.nan])
last_observed_hour = np.full(len(example_hours), np.nan)
latest_measurement_hour = np.nan
for index, hour in enumerate(example_hours):
    if example_observed[index]:
        latest_measurement_hour = hour
    last_observed_hour[index] = latest_measurement_hour
measurement_age = example_hours - last_observed_hour
never_observed = ~np.isfinite(last_observed_hour)
measurement_age[never_observed] = np.nan

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True, layout='constrained')
axes[0].plot(
    example_hours[example_observed], example_values[example_observed],
    linestyle='none', marker='o', markersize=8, color='#176b87',
)
axes[0].set_ylabel('Observed value')
axes[0].set_title('Observed measurements (hours 3, 6, and 9)')
axes[0].grid(alpha=0.2)

axes[1].plot(
    example_hours, measurement_age, marker='o', linewidth=2, color='#d97824',
)
axes[1].scatter(
    example_hours[never_observed], np.zeros(never_observed.sum()),
    marker='x', s=70, color='#a83232', zorder=3,
)
axes[1].axvspan(0.5, 2.5, color='#a83232', alpha=0.08)
axes[1].set_ylabel('Hours since seen')
axes[1].set_title('Measurement age Δ; red × = undefined before first observation')
axes[1].set_ylim(-0.6, 3.2)
axes[1].grid(alpha=0.2)
axes[1].annotate(
    'At hour 8: last seen at 6, so Δ = 8 - 6 = 2 h',
    xy=(8, 2), xytext=(4.1, 2.8),
    arrowprops={'arrowstyle': '->', 'color': '#333333'},
    fontsize=9,
)

axes[2].step(example_hours, never_observed.astype(int), where='mid', color='#a83232', linewidth=2)
axes[2].scatter(example_hours, never_observed.astype(int), color='#a83232', s=24)
axes[2].set(yticks=[0, 1], ylim=(-0.2, 1.2), ylabel='Never observed v', xlabel='ICU hour u')
axes[2].set_xticks(example_hours)
axes[2].grid(alpha=0.2)
plt.show()

print('Hour:            ', example_hours.tolist())
print('Observed:        ', example_observed.astype(int).tolist())
print('Last seen hour:  ', [None if not np.isfinite(hour) else int(hour) for hour in last_observed_hour])
print('Measurement age: ', [None if not np.isfinite(age) else int(age) for age in measurement_age])
print('Never observed:  ', never_observed.astype(int).tolist())

### Measurement age in the source-A training cohort

This applies the same causal calculation to every source-A training record and dynamic feature. At each `ICULOS` hour, the first heatmap shows median hours since the last observed value among records where that feature has been seen; the second shows the fraction of records whose feature has never been observed in the available record history. New observations reset age to zero, missing hours increase it, and never-seen entries remain undefined rather than being filled. The view is limited to the first 48 recorded ICU hours and excludes all labels. A record may begin after ICU hour 1, so “never observed” means not seen in the available record up to that hour.

In [ ]:
age_table = dataset.to_table(
    columns=['patient_id', 'hospital', 'ICULOS', *DYNAMIC],
    filter=(ds.field('hospital') == 'A') & ds.field('patient_id').isin(training_ids),
)
training_age_hours = age_table.to_pandas()
if set(training_age_hours['patient_id'].unique()) != set(training_ids):
    raise ValueError('Measurement-age scan does not match the source-A training records')

age_parts = []
for patient_id, record in training_age_hours.groupby('patient_id', sort=True):
    record = record.sort_values('ICULOS')
    iculos = record['ICULOS'].to_numpy(dtype=np.float64)
    measurements = record[DYNAMIC].to_numpy(dtype=np.float64)
    observed = np.isfinite(measurements)
    last_seen = np.maximum.accumulate(
        np.where(observed, iculos[:, None], -np.inf), axis=0
    )
    never = ~np.isfinite(last_seen)
    age = iculos[:, None] - last_seen
    age[never] = np.nan

    age_frame = pd.DataFrame(age, columns=DYNAMIC)
    age_frame['patient_id'] = patient_id
    age_frame['ICULOS'] = iculos
    age_long = age_frame.melt(
        id_vars=['patient_id', 'ICULOS'],
        var_name='feature',
        value_name='measurement_age',
    )
    never_frame = pd.DataFrame(never.astype(float), columns=DYNAMIC)
    never_frame['patient_id'] = patient_id
    never_frame['ICULOS'] = iculos
    never_long = never_frame.melt(
        id_vars=['patient_id', 'ICULOS'],
        var_name='feature',
        value_name='never_observed',
    )
    age_parts.append(age_long.merge(
        never_long,
        on=['patient_id', 'ICULOS', 'feature'],
        validate='one_to_one',
    ))

training_measurement_age = pd.concat(age_parts, ignore_index=True)
first_48 = training_measurement_age.loc[
    training_measurement_age['ICULOS'].between(1, 48)
]
icu_hours = list(range(1, 49))
median_age = (
    first_48.groupby(['feature', 'ICULOS'])['measurement_age']
    .median()
    .unstack('ICULOS')
    .reindex(index=DYNAMIC, columns=icu_hours)
)
never_fraction = (
    first_48.groupby(['feature', 'ICULOS'])['never_observed']
    .mean()
    .unstack('ICULOS')
    .reindex(index=DYNAMIC, columns=icu_hours)
)
print(
    f"Computed causal measurement age for {len(training_ids)} source-A training records "
    f"({len(training_age_hours):,} hourly rows)."
)

fig, axes = plt.subplots(1, 2, figsize=(18, 9), layout='constrained', sharey=True)
age_image = axes[0].imshow(
    median_age.to_numpy(dtype=np.float64),
    aspect='auto', interpolation='nearest', cmap='magma',
)
axes[0].set_title('Median hours since last measurement')
axes[0].set_xlabel('ICULOS (hour since ICU admission)')
axes[0].set_ylabel('Dynamic feature')
axes[0].set_xticks(np.arange(0, len(icu_hours), 4), labels=icu_hours[::4])
axes[0].set_yticks(np.arange(len(DYNAMIC)), labels=DYNAMIC)
fig.colorbar(age_image, ax=axes[0], label='Hours; blank means no prior observations')

never_image = axes[1].imshow(
    never_fraction.to_numpy(dtype=np.float64),
    aspect='auto', interpolation='nearest', cmap='Blues', vmin=0, vmax=1,
)
axes[1].set_title('Fraction never observed in available record history')
axes[1].set_xlabel('ICULOS (hour since ICU admission)')
axes[1].set_xticks(np.arange(0, len(icu_hours), 4), labels=icu_hours[::4])
axes[1].set_yticks(np.arange(len(DYNAMIC)), labels=DYNAMIC)
fig.colorbar(never_image, ax=axes[1], label='Fraction of records (0-1)')
fig.suptitle('Measurement age from source-A training records; no future measurements used')
plt.show()